In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("RDD Basics").getOrCreate()
sc = spark.sparkContext

In [2]:
numbers = [10, 20, 30, 40, 50]
rdd = sc.parallelize(numbers)
print(rdd)

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297


In [3]:
print(rdd.collect())

[10, 20, 30, 40, 50]


In [4]:
data = ["Rahul", "Sneha", "Arjun", "Priya"]
name_rdd = sc.parallelize(data)
name_rdd.collect()

['Rahul', 'Sneha', 'Arjun', 'Priya']

In [7]:
rdd.getNumPartitions()

2

In [8]:
rdd = sc.parallelize(numbers, 3)
rdd.getNumPartitions()

3

In [10]:
rdd = sc.parallelize([10, 20,30, 40, 50, 60], 3)
rdd.glom().collect()

[[10, 20], [30, 40], [50, 60]]

In [11]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
mapped_rdd = rdd.map(lambda x: x * 2)
filtered_rdd = mapped_rdd.filter(lambda x: x > 5)
print("Nothing executed yet")

Nothing executed yet


In [12]:
filtered_rdd.collect()

[6, 8, 10]

Lazy evaluation is when operations are stored and executed only when an action is called.

Map it will allow to iterate


In [13]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
squared_rdd = rdd.map(lambda x: x * x)
squared_rdd.collect()

[1, 4, 9, 16, 25]

In [14]:
rdd = sc.parallelize(["rahul", "sneha", "arjun"])
upper_rdd = rdd.map(lambda x: x.upper())
upper_rdd.collect()

['RAHUL', 'SNEHA', 'ARJUN']

filter..it filters and give by our condition

In [15]:
rdd = sc.parallelize([10, 15, 20, 25, 30])
even_rdd = rdd.filter(lambda x: x % 2 == 0)
even_rdd.collect()

[10, 20, 30]

In [16]:
rdd = sc.parallelize(["apple", "banana", "mango", "kiwi"])
long_words = rdd.filter(lambda x: len(x) > 5)
long_words.collect()

['banana']

flat map...can execute or split the words in the statement

In [19]:
lines = sc.parallelize([
    "santhosh is a data engineer",
    "spark sql pyspark"
])
words = lines.flatMap(lambda line: line.split(" "))
words.collect()


['santhosh', 'is', 'a', 'data', 'engineer', 'spark', 'sql', 'pyspark']

map applies function to each element but keeps output inside list (nested)

flat map applies function
then makes output into single list

In [21]:
lines = sc.parallelize(["hello world", "pyspark rdd"])
result = lines.map(lambda x: x.split(" "))
result.collect()

[['hello', 'world'], ['pyspark', 'rdd']]

In [20]:
lines = sc.parallelize(["hello world", "pyspark rdd"])
result = lines.flatMap(lambda x: x.split(" "))
result.collect()

['hello', 'world', 'pyspark', 'rdd']

Removes duplicate values from RDD
and Keeps only unique elements

In [22]:
rdd = sc.parallelize([10, 20, 20, 30, 30, 30, 40])
rdd.distinct().collect()

[10, 20, 30, 40]

In [23]:
rdd1 = sc.parallelize([1, 2, 3])
rdd2 = sc.parallelize([4, 5, 6])
rdd1.union(rdd2).collect()

[1, 2, 3, 4, 5, 6]

intersection-printing the common elements

In [25]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = sc.parallelize([4, 5, 6, 7, 8])
rdd1.intersection(rdd2).collect()

[4, 5]

subtract-eliminating the common elements and printing diff elements

In [26]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = sc.parallelize([4, 5, 6, 7, 8])
rdd1.subtract(rdd2).collect()

[1, 2, 3]

Actions are operations that actually execute the Spark job
They return result to your system

In [27]:
rdd = sc.parallelize([10, 20, 30])
rdd.collect()

[10, 20, 30]

In [28]:
rdd = sc.parallelize([10, 20, 30, 40])
rdd.count()

4

In [29]:
rdd = sc.parallelize([10, 20, 30])
rdd.first()

10

In [30]:
rdd = sc.parallelize([10, 20, 30, 40, 50])
rdd.take(3)

[10, 20, 30]

reduce means making all values into single value by the condition

In [31]:
rdd = sc.parallelize([10, 20, 30, 40])
rdd.reduce(lambda x, y: x + y)

100

reduceByKey-combines values of same key using aggregation (like sum)
groupByKey- groups all values of same key into a list

In [32]:
data = [
    ("IT", 70000),
    ("HR", 60000),
    ("IT", 75000),
    ("Finance", 80000),
    ("HR", 58000)
]
rdd = sc.parallelize(data)
rdd.collect()

[('IT', 70000),
 ('HR', 60000),
 ('IT', 75000),
 ('Finance', 80000),
 ('HR', 58000)]

In [33]:
rdd.reduceByKey(lambda a, b: a + b).collect()

[('IT', 145000), ('HR', 118000), ('Finance', 80000)]

In [34]:
grouped = rdd.groupByKey()
[(k, list(v)) for k, v in grouped.collect()]

[('IT', [70000, 75000]), ('HR', [60000, 58000]), ('Finance', [80000])]

mapValues applies operation only on values (not keys)
sortBy sorts elements based on a given condition

In [35]:
rdd = sc.parallelize([
    ("Rahul", 85),
    ("Sneha", 92),
    ("Arjun", 78)
])
rdd.mapValues(lambda x: x + 5).collect()

[('Rahul', 90), ('Sneha', 97), ('Arjun', 83)]

In [36]:
rdd2 = sc.parallelize([10, 40, 20, 30])
rdd2.sortBy(lambda x: x).collect()

[10, 20, 30, 40]

Word Count = split text assign (word,1) sum counts using reduceByKey

In [37]:
lines = sc.parallelize([
    "Santhosh is a data engineer",
    "Santhosh is a music Producer"
    "Santhosh is a hexaware employee"
])

word_counts = (
    lines
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

word_counts.collect()

[('Santhosh', 2),
 ('music', 1),
 ('employee', 1),
 ('is', 3),
 ('a', 3),
 ('data', 1),
 ('engineer', 1),
 ('ProducerSanthosh', 1),
 ('hexaware', 1)]

filter-selects only elements that satisfy a given condition

In [38]:
employees = sc.parallelize([
    ("Rahul", 70000),
    ("Sneha", 60000),
    ("Arjun", 75000),
    ("Priya", 80000),
    ("Karan", 50000)
])
high_salary = employees.filter(lambda x: x[1] > 65000)
high_salary.collect()

[('Rahul', 70000), ('Arjun', 75000), ('Priya', 80000)]